In [0]:
CREATE OR REPLACE TABLE dev.bronze.orders AS
SELECT * FROM VALUES
    (1, '2025-01-10', 101, 'Y'),
    (2, '2025-01-11', 102, 'N'),
    (3, '2025-01-12', NULL, 'Y'),
    (4, '2025-01-13', 104, 'X'),
    (5, '2020-05-10', 105, 'N'),
    (6, '2025-01-15', 106, 'Y')
AS orders(order_id, order_timestamp, customer_id, notifications);


SELECT * 
FROM dev.bronze.orders
ORDER BY order_id;



CREATE OR REFRESH STREAMING TABLE dev.silver.orders_warn
(
    CONSTRAINT valid_orders_warn
    EXPECT (notifications IN ('Y', 'N'))
)
AS
SELECT
    order_id,
    CAST(order_timestamp AS TIMESTAMP) AS order_timestamp,
    customer_id,
    notifications
FROM STREAM dev.bronze.orders;



CREATE OR REFRESH STREAMING TABLE dev.silver.orders_drop
(
    CONSTRAINT valid_orders_drop
    EXPECT (customer_id IS NOT NULL)
    ON VIOLATION DROP ROW
)
AS
SELECT
    order_id,
    CAST(order_timestamp AS TIMESTAMP) AS order_timestamp,
    customer_id,
    notifications
FROM STREAM dev.bronze.orders;


CREATE OR REFRESH STREAMING TABLE dev.silver.orders_fail
(
    CONSTRAINT valid_orders_fail
    EXPECT (customer_id IS NOT NULL)
    ON VIOLATION FAIL UPDATE
)
AS
SELECT
    order_id,
    CAST(order_timestamp AS TIMESTAMP) AS order_timestamp,
    customer_id,
    notifications
FROM STREAM dev.bronze.orders;


